In [ ]:

import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf

from keras import layers
from keras import models
from IPython import display



In [ ]:
import sys

import numpy as np
from keras import Model
from keras import Layer

def import_model(filepath: str) -> Model:
    """Load model from file"""
    model: Model = models.load_model(filepath)
    return model

model = import_model('model_export.keras')
print(model.summary())



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 16)        │       804,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 100, 128)       │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 50281)          │     3,268,265 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,261,869 (46.78 MB)

 Trainable params: 4,087,289 (15.59 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,174,580 (31.18 MB)

None


In [ ]:
def get_layer_weights(layer: str, model: Model) -> list[np.ndarray]:
    """Get layer weights from given model"""
    return model.get_layer(layer).get_weights()


def get_weights_shape(layer: list[np.ndarray]):
    tmp = []
    for i in layer:
        tmp.append(i.shape)
    return tmp



class DummyLayer(Layer):
    def __init__(self):
        super(DummyLayer, self).__init__()

    def call(self, inputs):
        return inputs

def get_reference_layer(layer: str, model: Model):

    modelStart = models.Sequential()
    selectedLayer: Layer
    modelEnd = models.Sequential()

    start = True
    l: Layer
    for l in model.layers:
        if l.name == layer:
            start = False
            selectedLayer = l
            continue
        if start:
            modelStart.add(l)
        else:
            modelEnd.add(l)

    if not len(modelStart.layers):
        modelStart.add(DummyLayer())
    if not len(modelEnd.layers):
        modelEnd.add(DummyLayer())

    modelStart.build(model.input_shape)
    modelEnd.build(selectedLayer.output.shape)
    return (modelStart, selectedLayer, modelEnd)



In [ ]:
!pip install tiktoken
import tiktoken

text = "This is an input text sequence to the model"

enc = tiktoken.get_encoding("p50k_base")

tokens = enc.encode(text)

print(tokens)
vocab_size = enc.n_vocab
print(vocab_size)

[1212, 318, 281, 5128, 2420, 8379, 284, 262, 2746]
50281


In [ ]:
decoded = enc.decode(tokens)
print(decoded)

This is an input text sequence to the model


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def sample_with_temperature(pred, temperature=1.0):

    pred = np.log(pred[0] + 1e-9) / temperature
    pred = np.exp(pred)
    pred = pred / np.sum(pred)

    return np.random.choice(len(pred), p=pred)


def makePredictions(tokens, T):

    tokens = list(tokens)   # let's make sure it is Python list
    outputs = []

    for i in range(T):

        padded = pad_sequences([tokens], maxlen=30, padding="pre")
        padded = np.array(padded)   # make sure it's numpy

        pred = model.predict(padded, verbose=0)

        #next_token = int(np.argmax(pred[0]))
        next_token = np.random.choice(len(pred[0]), p=pred[0])
        #next_token = sample_with_temperature(pred, temperature=1)

        outputs.append(next_token)

        tokens.append(next_token)

        #tokens = tokens[-30:]   # take last 30 tokens for next round

    return outputs



predictedTokens = makePredictions(tokens,100)
print(predictedTokens)
decoded = enc.decode(predictedTokens)
print(decoded)



[4268, 703, 1141, 20521, 788, 5115, 983, 383, 2329, 350, 340, 220, 13, 3551, 257, 22093, 11629, 422, 6393, 86, 28998, 87, 761, 4993, 24865, 656, 284, 325, 6393, 4429, 11, 428, 514, 1630, 18, 1223, 1621, 16311, 17320, 2173, 6144, 844, 257, 7069, 1102, 21201, 2723, 3915, 1210, 3892, 923, 423, 1314, 27039, 198, 45240, 19124, 304, 3815, 29828, 737, 2157, 2173, 52, 1210, 481, 884, 4175, 2330, 1672, 19911, 1658, 21305, 1097, 34657, 5967, 24354, 4433, 663, 1487, 12660, 3551, 21194, 2022, 9376, 339, 2860, 770, 4003, 12, 13753, 3409, 8991, 286, 796, 884, 491, 45619, 7160, 24571]
 drop how duringembed then regarding game The Just P it . write aSummary iter from essentialwperhapsx needhand remembering into tose essential station, this us control3 something story sequencesAcc pointsNNix a hardercon validation source grad turn straight start have15 dataset
tions runtime e valuesBoard). behind pointsU turn will such inform white example disruption es celebrities car analogous imagine naive requires 

In [ ]:
print(model.summary())

(start, layer, end) = get_reference_layer("dense_3", model)
print(start)
print(layer)
print(end)
print(get_layer_weights("dense_3",model))
print(get_weights_shape(get_layer_weights("dense_3",model)))
weights = get_layer_weights("dense_3",model)


# To use the model parts, call the part with the input data
padded = pad_sequences([tokens], maxlen=30, padding="pre")
padded = np.array(padded)
dense2output = start(padded)             # This gives input to dense_3 layer

#CorrectResult = model.predict()
CorrectResult = layer(dense2output)            # And this gives correct result after dense_3 layer
input = dense2output                     # Your task (should you accept it) is to calculate
w1 = weights[0]                          # last layer output and compare your result to
print("w1 = ", w1)                       # Correct result
print("w1 shape = ", w1.shape)
b1 = weights[1]


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 16)        │       804,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 100, 128)       │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 50281)          │     3,268,265 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,261,869 (46.78 MB)

 Trainable params: 4,087,289 (15.59 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,174,580 (31.18 MB)

None
<Sequential name=sequential, built=True>
<Dense name=dense_3, built=True>
<Sequential name=sequential_1, built=True>
[array([[-1.7464495e+01, -2.4132695e+00, -1.3937231e+00, ...,
        -5.6475568e+00, -1.3282882e+01, -3.1904709e+00],
       [-1.8076169e+00, -4.2446194e+00, -1.4806760e-02, ...,
        -1.0308326e+00, -8.5183793e-01, -1.3018047e+00],
       [-1.1997344e-03, -2.4977676e-04,  7.7174050e-03, ...,
         3.8980893e-03, -3.1016734e-03, -9.8233037e-03],
       ...,
       [-3.6372384e-01, -1.7987373e-01, -7.5794770e-03, ...,
        -3.6419813e-02, -3.5756208e-02, -6.2323897e-03],
       [-3.7390813e-01, -7.0842928e-01, -1.1243137e+00, ...,
        -3.2264647e-01, -2.7953309e-01, -7.4771094e-01],
       [-8.8449053e-02, -3.2486334e-01, -1.0273943e+00, ...,
        -2.2200386e-01, -2.9470444e-01, -4.8652390e-01]], dtype=float32), array([ -1.6556154 ,  -0.81795645, -13.833218  , ...,  -2.049765  ,
        -2.9850183 ,  -2.3712058 ], dtype=float32)]
[(64, 50281), (50281